In [0]:
from pyspark.sql.functions import *

customers = spark.table("workspace.silver.customers_clean")
orders = spark.table("workspace.silver.orders_clean")

customer_sales_summary = (
    customers.alias("c")
    .join(
        orders.alias("o"),
        col("c.customer_id") == col("o.customer_id"),
        "left"
    )
    .groupBy(
        col("c.customer_id"),
        col("c.customer_name"),
        col("c.city"),
        col("c.state")
    )
    .agg(
        count("o.order_id").alias("total_orders"),
        sum("o.quantity").alias("total_quantity"),
        round(sum("o.total_amount"), 2).alias("total_revenue"),
        max("o.order_date").alias("last_order_date")
    )
)

display(customer_sales_summary)

customer_id,customer_name,city,state,total_orders,total_quantity,total_revenue,last_order_date
11,Abeer Kibe,Bangalore,KA,6,21,112365.16,2026-04-29
13,Vansha Rai,Delhi,DL,3,9,56046.6,2026-05-17
44,Noah Kashyap,Mumbai,MH,2,4,14837.4,2026-02-12
67,Pushti Saini,Kolkata,WB,7,25,92852.71,2026-05-26
73,Balveer Chopra,Ahmedabad,GJ,4,10,36212.54,2026-05-06
88,Neha Narayan,Kolkata,WB,4,13,65504.65,2026-07-18
110,Darsh Devi,Chennai,TN,7,25,102344.89,2026-06-09
135,Peter Keer,Bangalore,KA,7,18,74882.5,2026-06-27
165,Qarin Sachdeva,Pune,MH,10,38,173691.32,2026-05-04
168,Damini Viswanathan,Pune,MH,3,8,40261.42,2026-04-04


In [0]:
customer_sales_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.customer_sales_summary")

In [0]:
products = spark.table("workspace.silver.products_clean")
orders = spark.table("workspace.silver.orders_clean")

product_performance = (
    products.alias("p")
    .join(
        orders.alias("o"),
        col("p.product_id") == col("o.product_id"),
        "left"
    )
    .groupBy(
        col("p.product_id"),
        col("p.product_name"),
        col("p.category")
    )
    .agg(
        sum("o.quantity").alias("units_sold"),
        round(sum("o.total_amount"), 2).alias("revenue"),
        count("o.order_id").alias("total_orders")
    )
)

display(product_performance)

product_id,product_name,category,units_sold,revenue,total_orders
2071,Fire,Beauty,37,78765.6,12
2122,Rock,Books,31,226326.66,11
2575,Rock,Home,20,1308.6,9
2699,Year,Electronics,29,43156.35,11
2011,Newspaper,Clothing,43,333819.32,18
2054,Including,Electronics,45,441856.8,13
2204,Factor,Sports,44,208699.04,12
2730,Analysis,Sports,7,60248.51,4
2270,Cover,Books,38,180120.0,9
2431,Response,Home,18,170170.74,8


In [0]:
product_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.product_performance")

In [0]:
daily_sales = (
    orders.groupBy("order_date")
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_items"),
        round(sum("total_amount"), 2).alias("daily_revenue")
    )
    .orderBy("order_date")
)

display(daily_sales)

order_date,total_orders,total_items,daily_revenue
2024-07-24,56,156,773620.98
2024-07-25,87,257,1340737.06
2024-07-26,65,192,996995.55
2024-07-27,67,201,1101341.19
2024-07-28,64,147,701494.73
2024-07-29,75,236,1097604.29
2024-07-30,59,184,1044192.82
2024-07-31,75,233,1204466.22
2024-08-01,65,210,1111646.56
2024-08-02,67,210,1106097.51


In [0]:
daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.daily_sales_summary")